# SHAP Explanation Stability Evaluation for KKBOX Customer Churn
**Authors:** Bhumi Wadhwani & Ayushi Ralhan  
**Guidance:** Prof. Ajaj Khan | Medi-Caps University  
**Research Scope:** Evaluating cross-model consistency of Tree-SHAP explanations across Random Forest, XGBoost, and CatBoost on the KKBOX Churn dataset.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

# Set paths
ROOT = Path("..").resolve()
RESULTS = ROOT / "results"
print(f"Results directory: {RESULTS}")

## 1. Churn Model Prediction Performance
Confirming models function as viable churn predictors before assessing explanation stability.

In [ ]:
metrics_df = pd.read_csv(RESULTS / "model_metrics.csv")
display(metrics_df[metrics_df["split"].str.contains("test")])

## 2. Global Explanation Stability
- **Spearman Rank Correlation** across full feature rankings
- **Jaccard Similarity** of Top-K (K=5, 10, 15) features

In [ ]:
global_stab = pd.read_csv(RESULTS / "global_stability_results.csv")
display(global_stab)

spearman_df = pd.read_csv(RESULTS / "spearman_consistency_matrix.csv", index_col=0)
jaccard_df = pd.read_csv(RESULTS / "jaccard_top10_consistency_matrix.csv", index_col=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(spearman_df, annot=True, fmt=".3f", cmap="Blues", ax=axes[0], square=True)
axes[0].set_title("Spearman Rank Correlation Matrix")
sns.heatmap(jaccard_df, annot=True, fmt=".3f", cmap="Oranges", ax=axes[1], square=True)
axes[1].set_title("Top-10 Feature Jaccard Consistency")
plt.tight_layout()
plt.show()

## 3. Top Churn Drivers Identified Across Models (SHAP)

In [ ]:
shap_df = pd.read_csv(RESULTS / "shap_importance.csv", index_col=0)
top_12 = shap_df.mean(axis=1).sort_values(ascending=False).head(12)
display(shap_df.loc[top_12.index])

## 4. Subgroup Explanation Stability & Per-Customer Agreement

In [ ]:
subgroup_df = pd.read_csv(RESULTS / "subgroup_stability_summary.csv")
display(subgroup_df)

## 5. Retention Recommendation Concordance
Measuring whether differences in SHAP explanations translate into divergent retention interventions.

In [ ]:
with open(RESULTS / "recommendation_concordance_summary.json") as f:
    summary = json.load(f)
print("Recommendation Concordance:", json.dumps(summary, indent=2))

rec_df = pd.read_csv(RESULTS / "retention_recommendations.csv")
display(rec_df.head(10))